In [ ]:
pip install datasets

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ---------------------------------------- 559.1/559.1 kB 3.6 MB/s  0:00:00
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ---------------------------------------- 0.0/796.8 kB ? eta -:--:--
   ---------------------------------------- 796.8/796.8 kB 4.1 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.0 MB 4.5 MB/s eta 0:00:01
   -------------------- ------------------- 2.1/4.0 MB 5.1 MB/s eta 0:00:01
   --------------------------------- ------ 3.4/4.0 MB 5.6 MB


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
pip install sentence_transformers

   ---------------------------------------- 0.0/739.8 kB ? eta -:--:--
   ---------------------------- ----------- 524.3/739.8 kB 4.5 MB/s eta 0:00:01
   ---------------------------------------- 739.8/739.8 kB 4.6 MB/s  0:00:00
   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   -------------------- ------------------- 6.3/12.1 MB 30.7 MB/s eta 0:00:01
   ---------------------------------------- 12.1/12.1 MB 32.6 MB/s  0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 2.9/2.9 MB 58.4 MB/s  0:00:00
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   --- ------------------------------------ 10.0/124.1 MB 62.4 MB/s eta 0:00:02
   ----- ---------------------------------- 17.0/124.1 MB 44.4 MB/s eta 0:00:03
   ------- -------------------------------- 23.3/124.1 MB 40.3 MB/s eta 0:00:03
   --------- ------------------------------ 28.3/124.1 MB 34.7 MB/s eta 0:00:03
   --------- -


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Import library

In [50]:
from collections import Counter, defaultdict
from typing import List, Dict, Literal, Union
import re
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

CACHE_DIR = "./cache"

In [6]:
ds = load_dataset("UniverseTBD/arxiv-abstracts-large")

print(ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed'],
        num_rows: 2292057
    })
})


## Discover data

In [7]:
print(ds.shape)

{'train': (2292057, 14)}


In [ ]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed'],
        num_rows: 2292057
    })
})


In [ ]:
for i in range(3):
    print(f"Example {i+1}:")
    print(ds["train"][i]["abstract"])
    print(ds["train"][i]["categories"])
    print("---" * 20)

Example 1:
  A fully differential calculation in perturbative quantum chromodynamics is
presented for the production of massive photon pairs at hadron colliders. All
next-to-leading order perturbative contributions from quark-antiquark,
gluon-(anti)quark, and gluon-gluon subprocesses are included, as well as
all-orders resummation of initial-state gluon radiation valid at
next-to-next-to-leading logarithmic accuracy. The region of phase space is
specified in which the calculation is most reliable. Good agreement is
demonstrated with data from the Fermilab Tevatron, and predictions are made for
more detailed tests with CDF and DO data. Predictions are shown for
distributions of diphoton pairs produced at the energy of the Large Hadron
Collider (LHC). Distributions of the diphoton pairs from the decay of a Higgs
boson are contrasted with those produced from QCD processes at the LHC, showing
that enhanced sensitivity to the signal can be obtained with judicious
selection of events.

hep-p

In [8]:
df = ds["train"].to_pandas()

print(df)

                       id           submitter  \
0               0704.0001      Pavel Nadolsky   
1               0704.0002        Louis Theran   
2               0704.0003         Hongjun Pan   
3               0704.0004        David Callan   
4               0704.0005  Alberto Torchinsky   
...                   ...                 ...   
2292052  supr-con/9608008     Ruslan Prozorov   
2292053  supr-con/9609001  Durga P. Choudhury   
2292054  supr-con/9609002  Durga P. Choudhury   
2292055  supr-con/9609003   Hasegawa Yasumasa   
2292056  supr-con/9609004    Masanori Ichioka   

                                                   authors  \
0        C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...   
1                          Ileana Streinu and Louis Theran   
2                                              Hongjun Pan   
3                                             David Callan   
4                 Wael Abu-Shammala and Alberto Torchinsky   
...                                    

In [9]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 2292057 entries, 0 to 2292056
Data columns (total 14 columns):
 #   Column          Dtype        
---  ------          -----        
 0   id              str          
 1   submitter       str          
 2   authors         str          
 3   title           str          
 4   comments        str          
 5   journal-ref     str          
 6   doi             str          
 7   report-no       str          
 8   categories      str          
 9   license         str          
 10  abstract        str          
 11  versions        object       
 12  update_date     datetime64[s]
 13  authors_parsed  object       
dtypes: datetime64[s](1), object(2), str(11)
memory usage: 2.9+ GB
None


In [ ]:
print(df.shape)

(100, 14)


In [ ]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 2292057 entries, 0 to 2292056
Data columns (total 14 columns):
 #   Column          Dtype        
---  ------          -----        
 0   id              str          
 1   submitter       str          
 2   authors         str          
 3   title           str          
 4   comments        str          
 5   journal-ref     str          
 6   doi             str          
 7   report-no       str          
 8   categories      str          
 9   license         str          
 10  abstract        str          
 11  versions        object       
 12  update_date     datetime64[s]
 13  authors_parsed  object       
dtypes: datetime64[s](1), object(2), str(11)
memory usage: 244.8+ MB
None


## Split category

Do bài chỉ cần chia primary categories nên chỉ cần split phần đầu tiên của categories làm label

<primary_category.subject> <side_category.subject> <...>
=>  <primary_category>, <side_category>

Ví dụ:
physics.acc-ph hep-ex quant-ph
=> physics, hep-ex

Sau đó lấy 5 category: [‘astro-ph’, ‘cond-mat’, ‘cs’, ‘math’, ‘physics’] để train


In [36]:
all_categories = set(df.categories)
set_feature_names = set()
print(len(all_categories))
print(list(all_categories)[:5])


77849
['nlin.SI cond-mat.str-el', 'q-bio.QM physics.med-ph q-bio.OT', 'math.OC cs.CC cs.SY math.DS', 'astro-ph.IM astro-ph.GA astro-ph.HE gr-qc', 'cond-mat.dis-nn math.DS nlin.CD nlin.CG physics.comp-ph']


In [37]:
for category in all_categories:
    parts = category.split(' ')
    #print(parts)
    for part in parts:
        label = part.split('.')[0]
        #print(label)
        set_feature_names.add(label)

feature_names = sorted(set_feature_names)
print(len(feature_names))
print(feature_names)

38
['acc-phys', 'adap-org', 'alg-geom', 'ao-sci', 'astro-ph', 'atom-ph', 'bayes-an', 'chao-dyn', 'chem-ph', 'cmp-lg', 'comp-gas', 'cond-mat', 'cs', 'dg-ga', 'econ', 'eess', 'funct-an', 'gr-qc', 'hep-ex', 'hep-lat', 'hep-ph', 'hep-th', 'math', 'math-ph', 'mtrl-th', 'nlin', 'nucl-ex', 'nucl-th', 'patt-sol', 'physics', 'plasm-ph', 'q-alg', 'q-bio', 'q-fin', 'quant-ph', 'solv-int', 'stat', 'supr-con']


In [28]:
samples = []

filter_feature_names = ['astro-ph', 'cond-mat', 'cs', 'math', 'physics']

for s in ds['train']:
    if (len(s['categories'].split(' ')) != 1):
        continue
    
    cur_category = s['categories'].strip().split('.')[0]
    if (cur_category not in filter_feature_names):
        continue
    samples.append(s)
    
    if (len(samples) >= 1000):
        break
print(len(samples))
for i in range(5):
    print(samples[i])


1000
{'id': '0704.0003', 'submitter': 'Hongjun Pan', 'authors': 'Hongjun Pan', 'title': 'The evolution of the Earth-Moon system based on the dark matter field\n  fluid model', 'comments': '23 pages, 3 figures', 'journal-ref': None, 'doi': None, 'report-no': None, 'categories': 'physics.gen-ph', 'license': None, 'abstract': "  The evolution of Earth-Moon system is described by the dark matter field\nfluid model proposed in the Meeting of Division of Particle and Field 2004,\nAmerican Physical Society. The current behavior of the Earth-Moon system agrees\nwith this model very well and the general pattern of the evolution of the\nMoon-Earth system described by this model agrees with geological and fossil\nevidence. The closest distance of the Moon to Earth was about 259000 km at 4.5\nbillion years ago, which is far beyond the Roche's limit. The result suggests\nthat the tidal friction may not be the primary cause for the evolution of the\nEarth-Moon system. The average dark matter field f

In [33]:
preprocessed_samples = []
for s in samples:
    abstract = s['abstract']

    # Remove \n characters in the middle and leading/trailing spaces
    abstract = abstract.strip().replace("\n", " ")

    # Remove special characters
    abstract = re.sub(r'[^\w\s]', '', abstract)
    # Remove digits
    abstract = re.sub(r'\d+', '', abstract)
    # Remove extra spaces
    abstract = re.sub(r'\s+', ' ', abstract).strip()
    # Convert to lower case
    abstract = abstract.lower()

    # for the label, we only keep the first part
    parts = s['categories'].split(' ')
    category = parts[0].split('.')[0]

    preprocessed_samples.append({
    "abstract": abstract,
    "category": category})

for i in range(5):
    print(preprocessed_samples[i])

{'abstract': 'the evolution of earthmoon system is described by the dark matter field fluid model proposed in the meeting of division of particle and field american physical society the current behavior of the earthmoon system agrees with this model very well and the general pattern of the evolution of the moonearth system described by this model agrees with geological and fossil evidence the closest distance of the moon to earth was about km at billion years ago which is far beyond the roches limit the result suggests that the tidal friction may not be the primary cause for the evolution of the earthmoon system the average dark matter field fluid constant derived from earthmoon system data is x sm this model predicts that the marss rotation is also slowing with the angular acceleration rate about x rad s', 'category': 'physics'}
{'abstract': 'we show that a determinant of stirling cycle numbers counts unlabeled acyclic singlesource automata the proof involves a bijection from these au

## encode unique category

In [38]:
label_to_id = {label: i for i, label in enumerate(feature_names)}
id_to_label = {i: label for i, label in enumerate(feature_names)}

print("Label to ID mapping:")
for label, id_ in label_to_id.items():
    print(f"{label} --> {id_}")

Label to ID mapping:
acc-phys --> 0
adap-org --> 1
alg-geom --> 2
ao-sci --> 3
astro-ph --> 4
atom-ph --> 5
bayes-an --> 6
chao-dyn --> 7
chem-ph --> 8
cmp-lg --> 9
comp-gas --> 10
cond-mat --> 11
cs --> 12
dg-ga --> 13
econ --> 14
eess --> 15
funct-an --> 16
gr-qc --> 17
hep-ex --> 18
hep-lat --> 19
hep-ph --> 20
hep-th --> 21
math --> 22
math-ph --> 23
mtrl-th --> 24
nlin --> 25
nucl-ex --> 26
nucl-th --> 27
patt-sol --> 28
physics --> 29
plasm-ph --> 30
q-alg --> 31
q-bio --> 32
q-fin --> 33
quant-ph --> 34
solv-int --> 35
stat --> 36
supr-con --> 37


## split train data 80/20

In [43]:
X = [sample['abstract'] for sample in preprocessed_samples]
y = [label_to_id[sample['category']] for sample in preprocessed_samples]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(len(X_train),len( X_test))
print(len(y_train),len( y_test))

print(X_train[0])

800 200
800 200
the polarized neutron scattering in helimagnetic mnsi at low t reveals existence of a partially disordered chiral state at ambient pressure in the magnetic field applied along axis below the first order transition to the nonchiral ferromagnetic state this unexpected phenomenon is explained by the analysis of the spinwave spectrum we demonstrate that the square of the spinwave gap becomes negative under magnetic field applied along and but not along the direction it is a result of competition between the spinwave interaction and cubic anisotropy this negative sign means an instability of the spin wave spectrum for the helix and leads to a destruction of the helical order giving rise to the partially disordered state below the first order ferromagnetic transition


In [52]:
class EmbeddingVectorizer:
    def __init__(
        self,
        model_name: str = 'intfloat/multilingual-e5-base',
        normalize: bool = True
    ):
        self.model = SentenceTransformer(model_name)
        self.normalize = normalize

    def _format_inputs(
    self,
    texts: List[str],
    mode: Literal['query', 'passage']
    ) -> List[str]:
        if mode not in {"query", "passage"}:
            raise ValueError("Mode must be either 'query' or 'passage'")
        return [f"{mode}: {text.strip()}" for text in texts]

    def transform(
        self,
        texts: List[str],
        mode: Literal['query', 'passage'] = 'query'
    ) -> List[List[float]]:
        if mode == 'raw':
            inputs = texts
        else:
            inputs = self._format_inputs(texts, mode)

        embeddings = self.model.encode(inputs, normalize_embeddings= self.normalize)
        return embeddings.tolist()

    def transform_numpy(
    self,
    texts: List[str],
    mode: Literal['query', 'passage'] = 'query'
    ) -> np.ndarray:
        return np.array(self.transform(texts, mode=mode))

In [55]:
# bow
vectorizer_scaler = CountVectorizer()
X_train_vectorizer = vectorizer_scaler.fit_transform(X_train)
X_test_vectorizer = vectorizer_scaler.transform(X_test)

# tf-idf
tf_idf_scaler = TfidfVectorizer()
X_train_tf_idf = tf_idf_scaler.fit_transform(X_train)
X_test_tf_idf = tf_idf_scaler.transform(X_test)

# sentence embedding
sen_scaler = EmbeddingVectorizer()
X_train_sen = sen_scaler.transform_numpy(X_train)
X_test_sen = sen_scaler.transform(X_test)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5742.38it/s]


In [56]:
X_train_bow, X_test_bow = np.array(X_train_vectorizer.toarray()), np.array(X_test_vectorizer.toarray())
X_train_tfidf, X_test_tfidf = np.array(X_train_tf_idf.toarray()), np.array(X_test_tf_idf.toarray())
X_train_embeddings, X_test_embeddings = np.array(X_train_sen), np.array(X_test_tf_idf)

In [ ]:
def train_and_test_kmeans(X_train, y_train, X_test, y_test, n_clusters: int):
    kmeans = KMeans(n_clusters, random_state=42)
    cluster_ids = kmeans.fit_predict(X_train)
    #print(((cluster_ids)))
    
    cluster_to_label = {}
    for cluster_id in set(cluster_ids):  
        labels_in_cluster = [y_train[i] for i in range(len(y_train)) if cluster_ids[i] == cluster_id]
        most_common_label = Counter(labels_in_cluster).most_common(1)[0][0]
        cluster_to_label[cluster_id] = most_common_label
    
    # for key, value in cluster_to_label.items():
    #     print(f"{key}: {value}")
        
    test_cluster_ids = kmeans.predict(X_test)
    
    y_pred = [cluster_to_label[cluster_id] for cluster_id in test_cluster_ids]
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=[id_to_label[i] for i in range(len(id_to_label))], output_dict=True)

    return y_pred, accuracy, report

km_bow_labels, km_bow_accuracy, km_bow_report = train_and_test_kmeans(X_train_vectorizer, y_train, X_test_vectorizer, y_test, n_clusters=len(label_to_id))



ValueError: Number of classes, 5, does not match size of target_names, 38. Try specifying the labels parameter

In [100]:

km_bow_labels, km_bow_accuracy, km_bow_report = train_and_test_kmeans(X_train_vectorizer, y_train, X_test_vectorizer, y_test, n_clusters=len(label_to_id))

 # Print K-Means results
print("Accuracies for K-Means:")
print(f"Bag of Words: {km_bow_accuracy:.4f}")

0: 11
1: 4
2: 4
3: 4
4: 4
5: 4
6: 11
7: 4
8: 4
9: 4
10: 4
11: 4
12: 22
13: 11
14: 4
15: 4
16: 4
17: 4
18: 22
19: 4
20: 4
21: 4
22: 4
23: 4
24: 4
25: 4
26: 4
27: 12
28: 4
29: 4
30: 4
31: 4
32: 4
33: 4
34: 4
35: 11
36: 4
37: 22


ValueError: Number of classes, 5, does not match size of target_names, 38. Try specifying the labels parameter